# ATML PA3 · Task 1 · Steps 3-4: LoRA-4, LoRA-16 and OPLoRA-16

Runs end-to-end with **Run All** / **Save Version** on Kaggle (GPU **T4 x2**, Internet ON).
Attach the **Full FT notebook output** under **Add Input** so the shared baseline and earlier runs
are imported (otherwise the baseline is evaluated again at the end of the GPU 1 lane).

What it does:
1. clones the repo and installs dependencies
2. `task1.prepare`: same data / probe set / prompts as every Task 1 run (data hash is checked)
3. `task1.import_prior` for each run: baseline and other runs from attached outputs; a finished
   training of one of these runs (FP16 export in `task1_artifacts/exports/`) is reused, not redone
4. two lanes in parallel, each training one run at a time on its own GPU:
   * **GPU 0**: `lora16` (train, then evaluate) → `oplora` (train, then evaluate)
   * **GPU 1**: `lora4` (train, then evaluate)

   All adapter runs: frozen FP16 backbone, FP32 adapters on q/k/v/o/gate/up/down of every layer,
   lr 2e-4, dropout 0.05, random A / zero B, no adapter biases; LoRA-16 and OPLoRA share their
   initial factors. OPLoRA: k = 128, bases from the FP32 starting weights, projected factors are
   exported and merged. Every export is checked (merged vs. unmerged logits; OPLoRA constraints).
5. `task1.diagnostics` for each run: ρ_k, η, R(z) on `down_proj` (layers 0, 13, 27)
6. `task1.report` over everything in this notebook (incl. imported runs) →
   **`/kaggle/working/task1_adapters_report.zip`**

Expected time on T4 x2: about 6-7 h (GPU 0 lane: two runs of ~2 h training + ~1.2 h evaluation).
If a version fails, attach its output and run again: finished trainings and evaluations are reused.

In [ ]:
# ---- settings ---------------------------------------------------------------------------
REPO_URL   = "https://github.com/alihasnaink/finetune-ground-scale.git"
BRANCH     = "main"
CONFIG     = "task1/configs/lora16.yaml"
RUN        = "adapters"
SMOKE_TEST = False   # True = tiny end-to-end pipeline check (~15-20 min), separate output folder
PRIOR_SEARCH = "/kaggle/input"   # where attached outputs of earlier Task 1 notebooks are mounted

REPO_DIR = "/kaggle/working/repo"
if SMOKE_TEST:
    OUT_ROOT, SCRATCH = "/kaggle/working/task1_smoke", "/tmp/atml_task1_smoke"
    ARTIFACTS = "/kaggle/working/task1_smoke_artifacts"
    OVERRIDES = ["data.num_examples=256", "train.num_updates=8", "train.warmup_updates=2",
                 "train.checkpoint_updates=[4,8]", "eval.limit=16"]
else:
    OUT_ROOT, SCRATCH = "/kaggle/working/task1", "/tmp/atml_task1"
    ARTIFACTS = "/kaggle/working/task1_artifacts"   # FP16 exports + FP32 deltas (kept, not zipped)
    OVERRIDES = []
ZIP_PATH = f"/kaggle/working/task1_{RUN}{'_smoke' if SMOKE_TEST else ''}_report.zip"

In [ ]:
!nvidia-smi
import os, subprocess
if os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
print(subprocess.check_output(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], text=True))

In [ ]:
!pip install -q -r {REPO_DIR}/requirements-task1.txt
import torch, transformers, lm_eval
print("torch", torch.__version__, "| transformers", transformers.__version__, "| lm_eval", lm_eval.__version__)
N_GPU = torch.cuda.device_count()
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPU)])
assert N_GPU >= 1, "enable a GPU accelerator (T4 x2 recommended)"

In [ ]:
import sys, time
ENV = os.environ | {"T1_OUT_ROOT": OUT_ROOT, "T1_SCRATCH_ROOT": SCRATCH, "T1_ARTIFACTS_ROOT": ARTIFACTS,
                    "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false",
                    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
                    "TQDM_MININTERVAL": "30", "HF_HUB_DISABLE_PROGRESS_BARS": "1"}

def _cmd(module, args, config=None):
    cmd = [sys.executable, "-m", f"task1.{module}", "--config", config or CONFIG, *args]
    return cmd + (["--set", *OVERRIDES] if OVERRIDES else [])

def run(module, *args, gpu="0", config=None):
    """Run a task1 script in the foreground, streaming its output; raise on failure."""
    t0 = time.time()
    p = subprocess.Popen(_cmd(module, args, config), cwd=REPO_DIR, env=ENV | {"CUDA_VISIBLE_DEVICES": gpu},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"task1.{module} failed (exit {p.returncode})")
    print(f"--- task1.{module} finished in {(time.time() - t0) / 60:.1f} min")

def run_bg(module, *args, gpu="1", logfile, config=None):
    """Start a task1 script in the background on another GPU, logging to a file."""
    f = open(logfile, "w")
    return subprocess.Popen(_cmd(module, args, config), cwd=REPO_DIR, env=ENV | {"CUDA_VISIBLE_DEVICES": gpu},
                            stdout=f, stderr=subprocess.STDOUT, text=True)

os.makedirs(OUT_ROOT, exist_ok=True)

In [ ]:
# runs in this notebook and how they are spread over the GPUs
CONFIGS = {'lora4': 'task1/configs/lora4.yaml', 'lora16': 'task1/configs/lora16.yaml', 'oplora': 'task1/configs/oplora.yaml'}
LANES = {'0': ['lora16', 'oplora'], '1': ['lora4']}
if N_GPU < 2:   # single GPU: everything runs one after another on GPU 0
    LANES = {"0": [r for rs in LANES.values() for r in rs]}
print(LANES)

In [ ]:
# 1) data, probe set, qualitative prompts, model + eval-dataset downloads
run("prepare")
# 2) reuse results of earlier Task 1 notebooks attached under Add Input (data hash is checked)
for r, c in CONFIGS.items():
    run("import_prior", "--search", PRIOR_SEARCH, config=c)
HAVE_BASELINE = os.path.exists(f"{OUT_ROOT}/eval/baseline/metrics.json")
print("baseline", "imported" if HAVE_BASELINE else "missing -> evaluated at the end of the last lane")

In [ ]:
# 3) the lanes: each trains its runs one at a time (training skipped if already complete) and
#    evaluates each final model right after its training
import shlex

def lane_commands(runs, with_baseline):
    sets = (["--set", *OVERRIDES] if OVERRIDES else [])
    cmds = []
    for r in runs:
        c = CONFIGS[r]
        cmds.append([sys.executable, "-m", "task1.train", "--config", c, *sets])
        cmds.append([sys.executable, "-m", "task1.evaluate", "--config", c, "--conditions", f"{r}_final", *sets])
    if with_baseline:
        cmds.append([sys.executable, "-m", "task1.evaluate", "--config", CONFIGS[runs[0]],
                     "--conditions", "baseline", *sets])
    return "set -e\n" + "\n".join(shlex.join(x) for x in cmds)

gpus = sorted(LANES)
bg = {}
for g in gpus[1:]:   # background lanes (GPU 1)
    logf = f"{OUT_ROOT}/lane_gpu{g}.log"
    script = lane_commands(LANES[g], with_baseline=not HAVE_BASELINE and g == gpus[-1])
    bg[g] = (subprocess.Popen(["bash", "-c", script], cwd=REPO_DIR, env=ENV | {"CUDA_VISIBLE_DEVICES": g},
                              stdout=open(logf, "w"), stderr=subprocess.STDOUT, text=True), logf)
    print(f"lane GPU {g}: {LANES[g]} -> {logf}")

# foreground lane (GPU 0), streamed here
for r in LANES["0"]:
    run("train", gpu="0", config=CONFIGS[r])
    run("evaluate", "--conditions", f"{r}_final", gpu="0", config=CONFIGS[r])
if not HAVE_BASELINE and len(gpus) == 1:
    run("evaluate", "--conditions", "baseline", gpu="0")

In [ ]:
# 4) wait for the background lane(s) and show the end of their logs
for g, (p, logf) in bg.items():
    while p.poll() is None:
        time.sleep(30)
    print(f"===== lane GPU {g} (exit {p.returncode}) =====")
    print(open(logf).read()[-8000:])
    assert p.returncode == 0, f"lane on GPU {g} failed; see {logf}"

In [ ]:
# 5) weight-space and activation diagnostics for every run of this notebook
for r, c in CONFIGS.items():
    run("diagnostics", gpu="0", config=c)

In [ ]:
# 6) tables, figures, qualitative sheets and the downloadable zip (all runs present)
run("report", "--zip", ZIP_PATH)

In [ ]:
# quick look at the headline tables
from pathlib import Path
from IPython.display import Markdown, display, FileLink, Image
rep = Path(OUT_ROOT) / "report"
for t in ["benchmarks_primary", "gsm8k_behavior", "resources", "displacement", "l2sp_objective",
          "adapters", "adapter_subspace_by_module", "diagnostics_weight_space",
          "diagnostics_R_math_vs_nonmath"]:
    f = rep / "tables" / f"{t}.md"
    if f.exists():
        display(Markdown(f.read_text()))
for f in sorted((rep / "figures").glob("*.png")):
    display(Image(filename=str(f)))
display(FileLink(os.path.relpath(ZIP_PATH, "/kaggle/working")))

In [ ]:
# optional: free scratch space (recovery checkpoints are not part of the output)
# import shutil; shutil.rmtree(SCRATCH, ignore_errors=True)
!du -sh {OUT_ROOT} {ARTIFACTS} {ZIP_PATH} {SCRATCH} 2>/dev/null